# 02 - Clean weather and build calendar features
**Input:** `notebooks/data/NYC_Weather_2016_2022.csv` (Open-Meteo hourly archive, NYC)
**Output:** `data/interim/weather_hourly.csv`, `data/interim/calendar_daily.csv`

Weather and holidays are forecaster features; heat/rain flags also define the surge-day case studies.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
RAW, INTERIM, PROCESSED = ROOT / "data/raw", ROOT / "data/interim", ROOT / "data/processed"
NB_DATA = ROOT / "notebooks/data"
INTERIM.mkdir(parents=True, exist_ok=True); PROCESSED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

In [2]:
w = pd.read_csv(NB_DATA / "NYC_Weather_2016_2022.csv", parse_dates=["time"])
w.columns = [c.split(" (")[0] for c in w.columns]
print(w.shape, w.time.min(), "->", w.time.max())
print("duplicates:", w.time.duplicated().sum())
full = pd.date_range(w.time.min(), w.time.max(), freq="h")
print("missing hours:", len(full.difference(w.time)))
print(w.describe().round(1).T)

(59760, 10) 2016-01-01 00:00:00 -> 2022-10-25 23:00:00
duplicates: 0
missing hours: 0
                     count                 mean                  min                  25%                  50%                  75%                  max   std
time                 59760  2019-05-29 23:30:00  2016-01-01 00:00:00  2017-09-14 11:45:00  2019-05-29 23:30:00  2021-02-10 11:15:00  2022-10-25 23:00:00   NaN
temperature_2m     59592.0                 13.2                -18.3                  5.2                 13.5                 21.8                 35.5   9.8
precipitation      59592.0                  0.1                  0.0                  0.0                  0.0                  0.0                 19.2   0.5
rain               59592.0                  0.1                  0.0                  0.0                  0.0                  0.0                 19.2   0.5
cloudcover         59592.0                 45.4                  0.0                  7.0                 35.0         

In [3]:
w = w.drop_duplicates("time").set_index("time").reindex(full).rename_axis("time")
w = w.interpolate(limit=3)            # fill only short gaps
print("remaining NaN:", int(w.isna().sum().sum()), "| hours with no temperature:", int(w.temperature_2m.isna().sum()))
print("NOTE: file ends", w.index.max(), "- later incidents have no weather; model on dates up to this limit")
w["heat_index_proxy"] = w.temperature_2m          # placeholder: replace with true heat index if humidity is added
w.reset_index().to_csv(INTERIM / "weather_hourly.csv", index=False)

remaining NaN: 1458 | hours with no temperature: 162
NOTE: file ends 2022-10-25 23:00:00 - later incidents have no weather; model on dates up to this limit


## Calendar: weekday, US federal holidays, heat and rain days

In [4]:
from pandas.tseries.holiday import USFederalHolidayCalendar
days = pd.DataFrame({"date": pd.date_range("2016-01-01", "2022-12-31")})
hol = USFederalHolidayCalendar()
names = hol.holidays("2016-01-01", "2022-12-31", return_name=True)
days["is_holiday"] = days.date.isin(names.index)
days["holiday_name"] = days.date.map(names)
days["dow"] = days.date.dt.dayofweek
days["is_weekend"] = days.dow >= 5

daily = w.temperature_2m.resample("D").agg(["max", "min"]).rename(columns={"max": "tmax", "min": "tmin"})
daily["precip_mm"] = w.precipitation.resample("D").sum()
daily = daily.reset_index().rename(columns={"time": "date"})
days = days.merge(daily, on="date", how="left")
days["is_heat_day"] = days.tmax >= 32          # ~90F; surge-day definition (tunable)
days["is_heavy_rain"] = days.precip_mm >= 12.7  # ~0.5 in
print(days[["is_holiday", "is_heat_day", "is_heavy_rain"]].sum())
days.to_csv(INTERIM / "calendar_daily.csv", index=False)
days[days.is_heat_day].groupby(days.date.dt.year).size()

is_holiday        72
is_heat_day       66
is_heavy_rain    204
dtype: int64


date
2016    12
2017     6
2018     9
2019     9
2020     7
2021     9
2022    14
dtype: int64